> ### Import the Library

In [1]:
import numpy as np
import pandas as pd

pd.options.display.max_columns = 999
pd.options.display.float_format = "{:.2f}".format
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

> ### EDA !

In [2]:
df = pd.read_csv("dataset_2024_2025_updated_palestine_israel.csv")
df.head()

,score,self_text,subreddit,created_time,author_name,controversiality,ups,downs,user_is_verified,post_created_time,clean_text,side
0,134,Which is also ridiculous because we literally ...,worldnews,2025-04-27 23:59:11,Cutapotamus,0,134,0,True,2025-04-27 22:33:44,Which is also ridiculous because we literally ...,Unclear
1,1,Because Israel will kill everyone inside? We w...,IsraelPalestine,2025-04-27 23:58:56,MrNewVegas123,0,1,0,True,2025-04-27 09:42:09,Because Israel will kill everyone inside? We w...,Palestine
2,1,There were a lot of big protests hence why the...,IsraelPalestine,2025-04-27 23:58:31,Best-Anxiety-6795,0,1,0,True,2025-04-27 18:36:57,There were a lot of big protests hence why the...,Unclear
3,5,Just because the administration and echo chamb...,worldnews,2025-04-27 23:57:58,Interesting-Type-908,0,5,0,True,2025-04-27 22:33:44,Just because the administration and echo chamb...,Unclear
4,1,"Good for you, and boo on your friends/acquaint...",IsraelPalestine,2025-04-27 23:56:51,cloudedknife,0,1,0,True,2025-04-27 21:39:55,"Good for you, and boo on your friends/acquaint...",Palestine


- ### Check Duplicated Data

In [3]:
len(df.drop_duplicates()) / len(df)

1.0

> The dataset are safe from duplicated data

- ### Check Missing Value

In [4]:
df.isna().sum()

score                0
self_text            0
subreddit            0
created_time         0
author_name          0
controversiality     0
ups                  0
downs                0
user_is_verified     0
post_created_time    0
clean_text           0
side                 0
dtype: int64

> The dataset are safe from missing value

---
> ### Why not checking Outliers?

**The analysis primarily focuses on sentiment analysis using the `self_text` and `side` (Palestine or Israel) columns, checking for outliers in the numerical columns is not essential. Outliers are more important when working with predictive models that depend on numerical features, such as regression or classification models.**

---

> ### Continue to do Sentiment Analysis

- ### Text Cleansing

In [5]:
import re

def cleansing_text(x):
  # clean double whitespace
  out_text = ' '.join(x.split())

  # clean url
  out_text = re.sub(r"http\S+|www\S+|https\S+", 'http', out_text)

  # clean username
  out_text = re.sub(r"@\S+", '@user', out_text)

  return(out_text)


In [6]:
df['clean_text'] = df['self_text'].apply(cleansing_text)

df.tail()

,score,self_text,subreddit,created_time,author_name,controversiality,ups,downs,user_is_verified,post_created_time,clean_text,side
250782,-2,"Biden is strength. Trump is a paper tiger, a w...",worldnews,2025-01-20 13:10:37,Kind-Afternoon8399,1,-2,0,True,2025-01-20 12:59:15,"Biden is strength. Trump is a paper tiger, a w...",Unclear
250783,17,Friend? More like sugar daddy imo,worldnews,2025-01-20 13:09:32,-TheWill-,0,17,0,True,2025-01-20 12:59:15,Friend? More like sugar daddy imo,Unclear
250784,49,The Houthis' [flag and slogan](https://en.wiki...,worldnews,2025-01-20 13:09:02,if_it_is_in_a,0,49,0,True,2025-01-20 12:59:15,The Houthis' [flag and slogan](http plastered ...,Israel
250785,-57,Lol.\n\nI hate Trump - but he has his uses.\n\...,worldnews,2025-01-20 13:08:13,SlyRax_1066,0,-57,0,True,2025-01-20 12:59:15,Lol. I hate Trump - but he has his uses. No on...,Unclear
250786,15,This is a sideways step.. Houthis will only at...,worldnews,2025-01-20 13:03:56,Hefty-Relationship-8,0,15,0,True,2025-01-20 12:59:15,This is a sideways step.. Houthis will only at...,Israel


In [7]:
import tensorflow as tf
from transformers import pipeline

# sentiment analysis task with twitter roberta model
sentiment_pipeline = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment")

Device set to use cuda:0


In [8]:
import pandas as pd
from datasets import Dataset
from transformers import pipeline, AutoTokenizer
import torch

# Load your DataFrame (assuming df is already defined)
df['clean_text'] = df['clean_text'].fillna("")  # Replace NaN with empty strings

# Convert to HuggingFace Dataset
dataset = Dataset.from_pandas(df)

# Load tokenizer and set device
tokenizer = AutoTokenizer.from_pretrained("cardiffnlp/twitter-roberta-base-sentiment")
device = 0 if torch.cuda.is_available() else -1

# Initialize the sentiment analysis pipeline
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment",
    tokenizer=tokenizer,
    truncation=True,
    max_length=512,
    device=device
)

# Function to process batch
def process_batch(batch):
    try:
        results = sentiment_pipeline(batch['clean_text'])
        return {
            "sentiment": [result['label'] for result in results],
            "sentiment_score": [result['score'] for result in results]
        }
    except Exception as e:
        print(f"Error processing batch: {e}")
        return {
            "sentiment": ["ERROR"] * len(batch['clean_text']),
            "sentiment_score": [0.0] * len(batch['clean_text'])
        }

# Apply batch processing
dataset = dataset.map(process_batch, batched=True, batch_size=64)

# Convert back to DataFrame
df = dataset.to_pandas()

# Map labels to readable format
label_mapping = {
    'LABEL_0': 'Negative',
    'LABEL_1': 'Neutral',
    'LABEL_2': 'Positive'
}
df['sentiment'] = df['sentiment'].map(label_mapping)

# Save to CSV
df.to_csv('dataset_with_sentiment.csv', index=False)

print("Dataset has been updated and saved as dataset_with_sentiment.csv")
print(df.head())


Device set to use cuda:0


Map:   0%|          | 0/250787 [00:00<?, ? examples/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Dataset has been updated and saved as dataset_with_sentiment.csv
   score                                          self_text        subreddit  \
0    134  Which is also ridiculous because we literally ...        worldnews   
1      1  Because Israel will kill everyone inside? We w...  IsraelPalestine   
2      1  There were a lot of big protests hence why the...  IsraelPalestine   
3      5  Just because the administration and echo chamb...        worldnews   
4      1  Good for you, and boo on your friends/acquaint...  IsraelPalestine   

          created_time           author_name  controversiality  ups  downs  \
0  2025-04-27 23:59:11           Cutapotamus                 0  134      0   
1  2025-04-27 23:58:56         MrNewVegas123                 0    1      0   
2  2025-04-27 23:58:31     Best-Anxiety-6795                 0    1      0   
3  2025-04-27 23:57:58  Interesting-Type-908                 0    5      0   
4  2025-04-27 23:56:51          cloudedknife                 0  

In [9]:
# Set the maximum column width to None to show full text
pd.set_option('display.max_colwidth', None)

# Now when you call .head(), it will display the full text of the 'self_text' and 'clean_text' columns
df.head()

,score,self_text,subreddit,created_time,author_name,controversiality,ups,downs,user_is_verified,post_created_time,clean_text,side,sentiment,sentiment_score
0,134,"Which is also ridiculous because we literally have a free trade agreement with them which Trump negotiated and said it was the best trade deal ever made. They are so hyper focused on Canada having tariffs on a few goods and completely ignore the fact that the US also has tariffs to protect their local industries. But the more I talk to Trumpers, the truth is its proud ignorance because they refuse to learn anything past talking points.",worldnews,2025-04-27 23:59:11,Cutapotamus,0,134,0,True,2025-04-27 22:33:44,"Which is also ridiculous because we literally have a free trade agreement with them which Trump negotiated and said it was the best trade deal ever made. They are so hyper focused on Canada having tariffs on a few goods and completely ignore the fact that the US also has tariffs to protect their local industries. But the more I talk to Trumpers, the truth is its proud ignorance because they refuse to learn anything past talking points.",Unclear,Negative,0.71
1,1,"Because Israel will kill everyone inside? We were not discussing Gaza, but claiming Hamas is a threat to Gaza because you will genocide everyone to in there is maybe not such a good position to take.",IsraelPalestine,2025-04-27 23:58:56,MrNewVegas123,0,1,0,True,2025-04-27 09:42:09,"Because Israel will kill everyone inside? We were not discussing Gaza, but claiming Hamas is a threat to Gaza because you will genocide everyone to in there is maybe not such a good position to take.",Palestine,Negative,0.91
2,1,There were a lot of big protests hence why the US for a time shipping arms to Saudi Arabia over it.,IsraelPalestine,2025-04-27 23:58:31,Best-Anxiety-6795,0,1,0,True,2025-04-27 18:36:57,There were a lot of big protests hence why the US for a time shipping arms to Saudi Arabia over it.,Unclear,Negative,0.58
3,5,"Just because the administration and echo chambers like Fox News, NewsMax, and OAN repeat it... doesn't make it 1) True or 2) a 'good'idea. Annexing Canada (as if they'd want to be Americans) would create more problems than solve.",worldnews,2025-04-27 23:57:58,Interesting-Type-908,0,5,0,True,2025-04-27 22:33:44,"Just because the administration and echo chambers like Fox News, NewsMax, and OAN repeat it... doesn't make it 1) True or 2) a 'good'idea. Annexing Canada (as if they'd want to be Americans) would create more problems than solve.",Unclear,Negative,0.77
4,1,"Good for you, and boo on your friends/acquaintances!\n\nThe plain fact is that there are like 15million people in Israel, Gaza, and the West Bank. 2/3s of them are Israeli. They aren't Russian, or American, or Arab, or Turkish, or Egyptian, or Etheopian, or French, or German, or or or...they're Israeli. There isn't anywhere for Israelis to go but Israel, because it isn't a colonial state, it is a country all its own and it will fight, and kill, and imprison others to secure its continued existence. The other 1/3 of people in that region are what most people would like to be Palestinians, living in Palestine.\n\nWe can disagree and discuss many things without fully agreeing - on what the borders of Palestine should be. On facets of how IDF is prosecuting the war in Gaza, or how it handles security in the West Bank. On facets of settlements and settler violence issues. On many things.\n\nBut the root cause of this conflict, and the thing that cannot be disagreed about if there is ever going to be peace without genocide, is one simple fact: Israel exists, and its people aren't going to allow that to change. Refusing to talk to an Israeli is a tacit refusal to agree to that foundational fact, and it perpetuates conflict. \n\nAgain, good for you!",IsraelPalestine,2025-04-27 23:56:51,cloudedknife,0,1,0,True,2025-04-27 21:39:55,"Good for you, and boo on your friends/acquaintances! The plain fact is that there are like 15million people in I

- #### **"We removed `Unclear` statements as they are not relevant to the context of our project (Palestine–Israel)."**

> ### Use new dataset

In [10]:
data = pd.read_csv ("dataset_with_sentiment.csv")
#change column name from main_topic to "side"
data = data.rename(columns={"main_topic": "side"})
data.to_csv("dataset_with_sentiment.csv")
data.head()

,score,self_text,subreddit,created_time,author_name,controversiality,ups,downs,user_is_verified,post_created_time,clean_text,side,sentiment,sentiment_score
0,134,"Which is also ridiculous because we literally have a free trade agreement with them which Trump negotiated and said it was the best trade deal ever made. They are so hyper focused on Canada having tariffs on a few goods and completely ignore the fact that the US also has tariffs to protect their local industries. But the more I talk to Trumpers, the truth is its proud ignorance because they refuse to learn anything past talking points.",worldnews,2025-04-27 23:59:11,Cutapotamus,0,134,0,True,2025-04-27 22:33:44,"Which is also ridiculous because we literally have a free trade agreement with them which Trump negotiated and said it was the best trade deal ever made. They are so hyper focused on Canada having tariffs on a few goods and completely ignore the fact that the US also has tariffs to protect their local industries. But the more I talk to Trumpers, the truth is its proud ignorance because they refuse to learn anything past talking points.",Unclear,Negative,0.71
1,1,"Because Israel will kill everyone inside? We were not discussing Gaza, but claiming Hamas is a threat to Gaza because you will genocide everyone to in there is maybe not such a good position to take.",IsraelPalestine,2025-04-27 23:58:56,MrNewVegas123,0,1,0,True,2025-04-27 09:42:09,"Because Israel will kill everyone inside? We were not discussing Gaza, but claiming Hamas is a threat to Gaza because you will genocide everyone to in there is maybe not such a good position to take.",Palestine,Negative,0.91
2,1,There were a lot of big protests hence why the US for a time shipping arms to Saudi Arabia over it.,IsraelPalestine,2025-04-27 23:58:31,Best-Anxiety-6795,0,1,0,True,2025-04-27 18:36:57,There were a lot of big protests hence why the US for a time shipping arms to Saudi Arabia over it.,Unclear,Negative,0.58
3,5,"Just because the administration and echo chambers like Fox News, NewsMax, and OAN repeat it... doesn't make it 1) True or 2) a 'good'idea. Annexing Canada (as if they'd want to be Americans) would create more problems than solve.",worldnews,2025-04-27 23:57:58,Interesting-Type-908,0,5,0,True,2025-04-27 22:33:44,"Just because the administration and echo chambers like Fox News, NewsMax, and OAN repeat it... doesn't make it 1) True or 2) a 'good'idea. Annexing Canada (as if they'd want to be Americans) would create more problems than solve.",Unclear,Negative,0.77
4,1,"Good for you, and boo on your friends/acquaintances!\n\nThe plain fact is that there are like 15million people in Israel, Gaza, and the West Bank. 2/3s of them are Israeli. They aren't Russian, or American, or Arab, or Turkish, or Egyptian, or Etheopian, or French, or German, or or or...they're Israeli. There isn't anywhere for Israelis to go but Israel, because it isn't a colonial state, it is a country all its own and it will fight, and kill, and imprison others to secure its continued existence. The other 1/3 of people in that region are what most people would like to be Palestinians, living in Palestine.\n\nWe can disagree and discuss many things without fully agreeing - on what the borders of Palestine should be. On facets of how IDF is prosecuting the war in Gaza, or how it handles security in the West Bank. On facets of settlements and settler violence issues. On many things.\n\nBut the root cause of this conflict, and the thing that cannot be disagreed about if there is ever going to be peace without genocide, is one simple fact: Israel exists, and its people aren't going to allow that to change. Refusing to talk to an Israeli is a tacit refusal to agree to that foundational fact, and it perpetuates conflict. \n\nAgain, good for you!",IsraelPalestine,2025-04-27 23:56:51,cloudedknife,0,1,0,True,2025-04-27 21:39:55,"Good for you, and boo on your friends/acquaintances! The plain fact is that there are like 15million people in I

In [11]:
# Drop rows where the 'sentiment' column is 'Unclear'
data = data[data['sentiment'].str.lower() != 'unclear'].reset_index(drop=True)

In [12]:
print(data.columns)


Index(['score', 'self_text', 'subreddit', 'created_time', 'author_name',
       'controversiality', 'ups', 'downs', 'user_is_verified',
       'post_created_time', 'clean_text', 'side', 'sentiment',
       'sentiment_score'],
      dtype='object')


In [13]:
# Assuming df is your DataFrame
filtered_df = data[(data["side"] == "Unclear") & (data["sentiment"] == "Negative")]


In [14]:
filtered_df.head()

,score,self_text,subreddit,created_time,author_name,controversiality,ups,downs,user_is_verified,post_created_time,clean_text,side,sentiment,sentiment_score
0,134,"Which is also ridiculous because we literally have a free trade agreement with them which Trump negotiated and said it was the best trade deal ever made. They are so hyper focused on Canada having tariffs on a few goods and completely ignore the fact that the US also has tariffs to protect their local industries. But the more I talk to Trumpers, the truth is its proud ignorance because they refuse to learn anything past talking points.",worldnews,2025-04-27 23:59:11,Cutapotamus,0,134,0,True,2025-04-27 22:33:44,"Which is also ridiculous because we literally have a free trade agreement with them which Trump negotiated and said it was the best trade deal ever made. They are so hyper focused on Canada having tariffs on a few goods and completely ignore the fact that the US also has tariffs to protect their local industries. But the more I talk to Trumpers, the truth is its proud ignorance because they refuse to learn anything past talking points.",Unclear,Negative,0.71
2,1,There were a lot of big protests hence why the US for a time shipping arms to Saudi Arabia over it.,IsraelPalestine,2025-04-27 23:58:31,Best-Anxiety-6795,0,1,0,True,2025-04-27 18:36:57,There were a lot of big protests hence why the US for a time shipping arms to Saudi Arabia over it.,Unclear,Negative,0.58
3,5,"Just because the administration and echo chambers like Fox News, NewsMax, and OAN repeat it... doesn't make it 1) True or 2) a 'good'idea. Annexing Canada (as if they'd want to be Americans) would create more problems than solve.",worldnews,2025-04-27 23:57:58,Interesting-Type-908,0,5,0,True,2025-04-27 22:33:44,"Just because the administration and echo chambers like Fox News, NewsMax, and OAN repeat it... doesn't make it 1) True or 2) a 'good'idea. Annexing Canada (as if they'd want to be Americans) would create more problems than solve.",Unclear,Negative,0.77
5,243,I wish he would find another distraction.,worldnews,2025-04-27 23:56:46,crabmuncher,0,243,0,True,2025-04-27 22:33:44,I wish he would find another distraction.,Unclear,Negative,0.59
8,14,"The Republicans would probably carve up the country so that all the big cities are in one ""state"" and then create a hundred, tiny rural states that are typically conservative giving the conservative mini-states 100 times the state votes for Senate. They'd gerrymander the fuck out of Canada.",worldnews,2025-04-27 23:54:06,Ex-CultMember,0,14,0,True,2025-04-27 22:33:44,"The Republicans would probably carve up the country so that all the big cities are in one ""state"" and then create a hundred, tiny rural states that are typically conservative giving the conservative mini-states 100 times the state votes for Senate. They'd gerrymander the fuck out of Canada.",Unclear,Negative,0.71
